# 02 · Regresión Logística — Predicción de abandono académico

**Proyecto:** JARVIS_SANGRON  
**Asignatura:** Minería de Datos  
**Dataset:** OULAD — `studentInfo.csv`  
**Tarea:** Clasificación binaria supervisada

> **Objetivo:** estimar la probabilidad de abandono académico y construir una señal de alerta temprana que permita priorizar estudiantes en riesgo.

### Resultado final de referencia

| Elemento | Resultado |
|---|---:|
| Umbral seleccionado con validación | **0.27** |
| Recall de abandono en TEST | **0.7310** |
| F1 de abandono en TEST | **0.5110** |
| AUC en TEST | **0.6730** |

La clase positiva del problema es **abandono = 1**. El análisis prioriza **Recall** porque interesa reducir los falsos negativos: estudiantes que realmente abandonan pero no son identificados como riesgo.

## Ruta del análisis

La organización sigue la secuencia trabajada en clase:

1. Identificar la métrica y su valor actual.
2. Identificar la variable que permite calcular la métrica.
3. Definir el **target**.
4. Seleccionar variables explicativas.
5. Particionar los datos.
6. Entrenar el modelo.
7. Evaluar sobre entrenamiento.
8. Evaluar sobre datos no utilizados para entrenar.
9. Interpretar los resultados de cada iteración.

Además, al final se incorpora una mejora metodológica con **Train / Validation / Test** para seleccionar el umbral sin utilizar el conjunto de prueba.

## 1. Librerías y configuración

Las celdas de instalación se eliminan del flujo principal para mantener el notebook limpio. Si alguna librería falta, puede instalarse una sola vez desde la terminal o con `%pip install nombre_libreria`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    average_precision_score
)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titleweight'] = 'bold'

In [ ]:
def tabla_metricas(y_real, y_pred, y_prob, nombre='Modelo'):
    """Devuelve las métricas principales de clasificación en una tabla."""
    return pd.DataFrame({
        'Modelo': [nombre],
        'Accuracy': [accuracy_score(y_real, y_pred)],
        'Precision': [precision_score(y_real, y_pred, zero_division=0)],
        'Recall': [recall_score(y_real, y_pred, zero_division=0)],
        'F1': [f1_score(y_real, y_pred, zero_division=0)],
        'AUC': [roc_auc_score(y_real, y_prob)]
    }).set_index('Modelo')


def graficar_matriz_confusion(y_real, y_pred, titulo):
    """Muestra conteos y porcentaje por clase real."""
    matriz = confusion_matrix(y_real, y_pred)
    porcentajes = matriz / matriz.sum(axis=1, keepdims=True)
    etiquetas = np.array([
        [f'{matriz[i, j]}\n({porcentajes[i, j]:.1%})' for j in range(matriz.shape[1])]
        for i in range(matriz.shape[0])
    ])

    ax = sns.heatmap(
        matriz,
        annot=etiquetas,
        fmt='',
        cbar=False,
        xticklabels=['No abandono', 'Abandono'],
        yticklabels=['No abandono', 'Abandono']
    )
    ax.set_title(titulo)
    ax.set_xlabel('Predicción del modelo')
    ax.set_ylabel('Valor real')
    plt.tight_layout()
    plt.show()

    return matriz

## 2. Carga y comprensión del dataset

La unidad de análisis es un **registro estudiante–módulo–presentación**. El archivo `studentInfo.csv` contiene información académica, demográfica y contextual utilizada para construir el modelo.

In [ ]:
RUTA_DATOS = Path('../Data/studentInfo.csv')

df = pd.read_csv(RUTA_DATOS)

print(f'Registros : {df.shape[0]:,}')
print(f'Variables : {df.shape[1]}')
df.head()

In [ ]:
resumen_tipos = pd.DataFrame({
    'Tipo': df.dtypes.astype(str),
    'Nulos': df.isna().sum(),
    'Valores únicos': df.nunique(dropna=True)
})
resumen_tipos

## 3. Métrica principal y definición del target

La métrica de negocio observada es la **tasa de abandono**:

$$
\text{Tasa de abandono} = \frac{\text{Registros con Withdrawn}}{\text{Total de registros}} \times 100
$$

Se construye la variable objetivo:

$$
Y =
\begin{cases}
1, & \text{si final\_result = Withdrawn}\\
0, & \text{en otro caso}
\end{cases}
$$

`final_result` **no se utiliza como predictor**, porque se usa para construir el target y produciría **fuga de información (data leakage)**.

In [ ]:
df['abandono'] = (df['final_result'] == 'Withdrawn').astype(int)

distribucion = (
    df['abandono']
    .value_counts()
    .sort_index()
    .rename(index={0: 'No abandono', 1: 'Abandono'})
    .to_frame('Cantidad')
)
distribucion['Porcentaje'] = distribucion['Cantidad'] / len(df)
distribucion

In [ ]:
ax = distribucion['Cantidad'].plot(kind='bar', rot=0)
ax.set_title('Distribución de la variable objetivo')
ax.set_xlabel('Clase')
ax.set_ylabel('Cantidad de registros')

for i, (cantidad, porcentaje) in enumerate(zip(distribucion['Cantidad'], distribucion['Porcentaje'])):
    ax.text(i, cantidad, f'{porcentaje:.1%}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

### Lectura del problema

La clase abandono representa aproximadamente un tercio de los registros. Por ello, **Accuracy no debe analizarse de forma aislada**. Para una alerta temprana son especialmente importantes:

- **Precision:** de las alertas generadas, cuántas corresponden realmente a abandono.
- **Recall:** de los abandonos reales, cuántos logra detectar el modelo.
- **F1:** equilibrio entre Precision y Recall.
- **AUC:** capacidad general del modelo para discriminar entre ambas clases.

$$
\text{Precision}=\frac{TP}{TP+FP}
\qquad
\text{Recall}=\frac{TP}{TP+FN}
$$

$$
F1=2\cdot\frac{Precision\cdot Recall}{Precision+Recall}
$$

# 4. Primera iteración — modelo básico

Se comienza con dos variables explicativas para construir un **baseline interpretable**:

- `num_of_prev_attempts`: número de intentos previos.
- `studied_credits`: créditos estudiados.

La regresión logística estima:

$$
P(Y=1\mid X)=\frac{1}{1+e^{-z}},
\qquad
z=\beta_0+\beta_1X_1+\cdots+\beta_pX_p
$$

La probabilidad se convierte en clase utilizando inicialmente el umbral estándar **τ = 0.50**.

In [ ]:
y = df['abandono']
X = df[['num_of_prev_attempts', 'studied_credits']].copy()

pd.DataFrame({
    'Nulos en X': X.isna().sum(),
}).T

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=777,
    stratify=y
)

pd.DataFrame({
    'Conjunto': ['TRAIN', 'TEST'],
    'Registros': [len(X_train), len(X_test)],
    'Abandono (%)': [y_train.mean() * 100, y_test.mean() * 100]
})

In [ ]:
X_train_sm = sm.add_constant(X_train, has_constant='add')
modelo_1 = sm.Logit(y_train, X_train_sm).fit()

modelo_1.summary()

### Interpretación de coeficientes — primera iteración

Un coeficiente positivo aumenta los **log-odds** estimados de abandono; uno negativo los reduce. La significancia estadística se revisa mediante el **valor p**, utilizando como nivel de referencia **α = 0.05**.

In [ ]:
tabla_coef_1 = pd.DataFrame({
    'Coeficiente': modelo_1.params,
    'Odds Ratio': np.exp(modelo_1.params),
    'p-value': modelo_1.pvalues
})
tabla_coef_1

In [ ]:
y_prob_train = modelo_1.predict(X_train_sm)
y_pred_train = (y_prob_train >= 0.50).astype(int)

X_test_sm = sm.add_constant(X_test, has_constant='add')
y_prob_test = modelo_1.predict(X_test_sm)
y_pred_test = (y_prob_test >= 0.50).astype(int)

metricas_1 = pd.concat([
    tabla_metricas(y_train, y_pred_train, y_prob_train, 'Iteración 1 · TRAIN'),
    tabla_metricas(y_test, y_pred_test, y_prob_test, 'Iteración 1 · TEST')
])
metricas_1

In [ ]:
matriz_1 = graficar_matriz_confusion(
    y_test,
    y_pred_test,
    'Primera iteración · Matriz de confusión en TEST'
)

In [ ]:
accuracy_base = y_test.value_counts(normalize=True).max()

pd.DataFrame({
    'Referencia': ['Clasificador mayoritario', 'Regresión logística · Iteración 1'],
    'Accuracy': [accuracy_base, accuracy_score(y_test, y_pred_test)]
})

### Conclusión de la primera iteración

- **Accuracy TEST:** 0.6908, apenas por encima del baseline mayoritario de 0.6884.
- **Recall de abandono:** 0.0635.
- **AUC:** 0.6015.

De **2.031 abandonos reales**, el modelo detectó solamente **129**. La primera iteración es útil como punto de partida, pero no como sistema de alerta temprana. Esto justifica incorporar más información disponible en OULAD.

# 5. Segunda iteración — incorporación de más variables

Se incorporan variables académicas, demográficas y contextuales disponibles antes del resultado final. Las variables categóricas se convierten en variables dummy mediante **one-hot encoding** con `drop_first=True`.

In [ ]:
variables_segunda_iteracion = [
    'code_module',
    'code_presentation',
    'gender',
    'region',
    'highest_education',
    'imd_band',
    'age_band',
    'num_of_prev_attempts',
    'studied_credits',
    'disability'
]

X2 = df[variables_segunda_iteracion].copy()

nulos_antes = X2.isna().sum().sort_values(ascending=False)
nulos_antes[nulos_antes > 0]

In [ ]:
X2['imd_band'] = X2['imd_band'].fillna('Unknown')

columnas_categoricas = X2.select_dtypes(include='str').columns

X2 = pd.get_dummies(
    X2,
    columns=columnas_categoricas,
    drop_first=True,
    dtype=int
)

print(f'Variables utilizadas después de codificar: {X2.shape[1]}')
print(f'Valores nulos restantes: {X2.isna().sum().sum()}')
X2.head()

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2,
    y,
    test_size=0.20,
    random_state=777,
    stratify=y
)

X2_train_sm = sm.add_constant(X2_train, has_constant='add')
modelo_2 = sm.Logit(y2_train, X2_train_sm).fit(maxiter=200)

print(f'Convergió    : {modelo_2.mle_retvals.get("converged", False)}')
print(f'Pseudo R²   : {modelo_2.prsquared:.5f}')
print(f'LLR p-value : {modelo_2.llr_pvalue:.3e}')

In [ ]:
tabla_coef_2 = pd.DataFrame({
    'Coeficiente': modelo_2.params,
    'Odds Ratio': np.exp(modelo_2.params),
    'p-value': modelo_2.pvalues
})

significativas_2 = (
    tabla_coef_2
    .drop(index='const')
    .query('`p-value` < 0.05')
    .sort_values('p-value')
)

print(f'Predictores/categorías significativas (p < 0.05): {len(significativas_2)}')
significativas_2.head(15)

> **Nota de interpretación:** en las variables categóricas, cada coeficiente se interpreta **respecto a la categoría de referencia omitida** por `drop_first=True`. Los coeficientes describen asociaciones dentro del modelo; no demuestran causalidad.

In [ ]:
X2_test_sm = sm.add_constant(X2_test, has_constant='add')
y2_prob_test = modelo_2.predict(X2_test_sm)
y2_pred_test = (y2_prob_test >= 0.50).astype(int)

metricas_2 = tabla_metricas(
    y2_test,
    y2_pred_test,
    y2_prob_test,
    'Iteración 2 · TEST · τ=0.50'
)
metricas_2

In [ ]:
comparacion_12 = pd.concat([
    tabla_metricas(y_test, y_pred_test, y_prob_test, 'Iteración 1'),
    tabla_metricas(y2_test, y2_pred_test, y2_prob_test, 'Iteración 2')
])
comparacion_12

In [ ]:
ax = comparacion_12[['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']].plot(kind='bar', rot=0)
ax.set_title('Comparación de las dos iteraciones · Umbral 0.50')
ax.set_ylabel('Valor de la métrica')
ax.set_ylim(0, 1)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=5)
plt.tight_layout()
plt.show()

### Lectura de la segunda iteración

La segunda iteración mejora la discriminación del modelo:

- **AUC:** 0.6015 → **0.6763**.
- **Recall de abandono:** 0.0635 → **0.1649**.
- **F1:** 0.1135 → **0.2582**.

Aun así, con **τ = 0.50** el modelo continúa dejando sin detectar una gran cantidad de abandonos. Por ello se analiza el efecto del umbral.

# 6. Efecto del umbral de clasificación

La regla de decisión es:

$$
\hat{Y}=
\begin{cases}
1, & \hat{p}\geq\tau\\
0, & \hat{p}<\tau
\end{cases}
$$

Al reducir τ, normalmente aumenta el **Recall**, pero también aumentan los falsos positivos y puede disminuir la **Precision**.

> Esta sección es **exploratoria**. La selección formal del umbral se realizará después con un conjunto de **validación**, no con TEST.

In [ ]:
resultados_umbral = []

for umbral in np.arange(0.20, 0.51, 0.05):
    pred = (y2_prob_test >= umbral).astype(int)
    resultados_umbral.append({
        'Umbral': round(float(umbral), 2),
        'Accuracy': accuracy_score(y2_test, pred),
        'Precision': precision_score(y2_test, pred, zero_division=0),
        'Recall': recall_score(y2_test, pred, zero_division=0),
        'F1': f1_score(y2_test, pred, zero_division=0)
    })

tabla_umbrales = pd.DataFrame(resultados_umbral)
tabla_umbrales

In [ ]:
ax = tabla_umbrales.set_index('Umbral')[['Precision', 'Recall', 'F1']].plot(marker='o')
ax.set_title('Efecto del umbral sobre Precision, Recall y F1')
ax.set_xlabel('Umbral de clasificación (τ)')
ax.set_ylabel('Valor de la métrica')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

La exploración muestra por qué el umbral estándar de 0.50 no es necesariamente apropiado para una alerta temprana. Sin embargo, **no se reporta este ajuste como resultado final**, porque TEST debe reservarse para la evaluación final.

# 7. Evaluación final — Train / Validation / Test

Para evitar utilizar TEST en decisiones de ajuste, se realiza una separación final:

| Conjunto | Uso | Proporción |
|---|---|---:|
| **TRAIN** | Estimar los coeficientes del modelo | 70 % |
| **VALIDATION** | Seleccionar el umbral **τ** | 15 % |
| **TEST** | Evaluación final, una sola vez | 15 % |

In [ ]:
X3_train, X3_temp, y3_train, y3_temp = train_test_split(
    X2,
    y,
    test_size=0.30,
    random_state=777,
    stratify=y
)

X3_val, X3_test, y3_val, y3_test = train_test_split(
    X3_temp,
    y3_temp,
    test_size=0.50,
    random_state=777,
    stratify=y3_temp
)

resumen_particiones = pd.DataFrame({
    'Conjunto': ['TRAIN', 'VALIDATION', 'TEST'],
    'Registros': [len(X3_train), len(X3_val), len(X3_test)],
    'Proporción': [len(X3_train)/len(X2), len(X3_val)/len(X2), len(X3_test)/len(X2)],
    'Abandono (%)': [y3_train.mean()*100, y3_val.mean()*100, y3_test.mean()*100]
})
resumen_particiones

In [ ]:
ax = resumen_particiones.set_index('Conjunto')['Abandono (%)'].plot(kind='bar', rot=0)
ax.set_title('Proporción de abandono por partición')
ax.set_ylabel('Abandono (%)')
plt.tight_layout()
plt.show()

In [ ]:
X3_train_sm = sm.add_constant(X3_train, has_constant='add')
modelo_final = sm.Logit(y3_train, X3_train_sm).fit(maxiter=200)

print(f'Convergió    : {modelo_final.mle_retvals.get("converged", False)}')
print(f'Pseudo R²   : {modelo_final.prsquared:.5f}')
print(f'LLR p-value : {modelo_final.llr_pvalue:.3e}')

## 7.1. Selección del umbral con VALIDATION

El criterio seleccionado es **maximizar F1** sobre VALIDATION. De esta manera, TEST permanece sin utilizar durante el ajuste.

In [ ]:
X3_val_sm = sm.add_constant(X3_val, has_constant='add')
y3_prob_val = modelo_final.predict(X3_val_sm)

resultados_validacion = []

for umbral in np.arange(0.10, 0.61, 0.01):
    pred_val = (y3_prob_val >= umbral).astype(int)
    resultados_validacion.append({
        'Umbral': round(float(umbral), 2),
        'Accuracy': accuracy_score(y3_val, pred_val),
        'Precision': precision_score(y3_val, pred_val, zero_division=0),
        'Recall': recall_score(y3_val, pred_val, zero_division=0),
        'F1': f1_score(y3_val, pred_val, zero_division=0)
    })

tabla_validacion = pd.DataFrame(resultados_validacion)
mejor_validacion = tabla_validacion.loc[tabla_validacion['F1'].idxmax()]
umbral_final = float(mejor_validacion['Umbral'])

pd.DataFrame(mejor_validacion).T.rename(index={mejor_validacion.name: 'Mejor configuración'})

In [ ]:
ax = tabla_validacion.set_index('Umbral')[['Precision', 'Recall', 'F1']].plot()
ax.axvline(umbral_final, linestyle='--', label=f'Umbral elegido = {umbral_final:.2f}')
ax.set_title('Selección del umbral con VALIDATION')
ax.set_xlabel('Umbral de clasificación (τ)')
ax.set_ylabel('Valor de la métrica')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

## 7.2. Evaluación final sobre TEST

Una vez elegido el umbral, **no se modifica nuevamente**. Se aplica directamente al conjunto TEST para obtener la estimación final de desempeño.

In [ ]:
X3_test_sm = sm.add_constant(X3_test, has_constant='add')
y3_prob_test = modelo_final.predict(X3_test_sm)
y3_pred_test = (y3_prob_test >= umbral_final).astype(int)

metricas_finales = tabla_metricas(
    y3_test,
    y3_pred_test,
    y3_prob_test,
    f'Modelo final · TEST · τ={umbral_final:.2f}'
)
metricas_finales

In [ ]:
matriz_final = graficar_matriz_confusion(
    y3_test,
    y3_pred_test,
    f'Modelo final · TEST · Umbral {umbral_final:.2f}'
)

print('Matriz de confusión:')
print(matriz_final)

In [ ]:
print(classification_report(
    y3_test,
    y3_pred_test,
    target_names=['No abandono', 'Abandono'],
    zero_division=0
))

### Cómo leer la matriz final

Para la clase **Abandono**:

- **TP = 1.114:** abandonos detectados correctamente.
- **FN = 410:** abandonos que el modelo no detectó.
- **FP = 1.722:** alertas generadas para estudiantes que finalmente no abandonaron.

El modelo final detecta aproximadamente **73 de cada 100 abandonos reales**, a cambio de generar una cantidad considerable de falsas alertas.

## 7.3. Curvas de evaluación

La curva **ROC** resume la capacidad general de separación entre clases. La curva **Precision–Recall** es especialmente informativa cuando el interés se concentra en la clase positiva y existe desbalance.

In [ ]:
fpr, tpr, _ = roc_curve(y3_test, y3_prob_test)
precision_curve, recall_curve, _ = precision_recall_curve(y3_test, y3_prob_test)

fig, ax = plt.subplots()
ax.plot(fpr, tpr, label=f'ROC · AUC = {roc_auc_score(y3_test, y3_prob_test):.3f}')
ax.plot([0, 1], [0, 1], linestyle='--', label='Referencia aleatoria')
ax.set_title('Curva ROC · Modelo final')
ax.set_xlabel('Tasa de falsos positivos')
ax.set_ylabel('Tasa de verdaderos positivos')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
ap = average_precision_score(y3_test, y3_prob_test)

fig, ax = plt.subplots()
ax.plot(recall_curve, precision_curve, label=f'Average Precision = {ap:.3f}')
ax.axhline(y3_test.mean(), linestyle='--', label=f'Prevalencia = {y3_test.mean():.3f}')
ax.set_title('Curva Precision–Recall · Modelo final')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend()
plt.tight_layout()
plt.show()

# 8. Resumen comparativo

La tabla siguiente concentra la evolución del modelo. La comparación entre las iteraciones de 80/20 sirve para observar el efecto de incorporar variables; el modelo final utiliza la partición metodológicamente más rigurosa 70/15/15.

In [ ]:
resumen_modelos = pd.concat([
    tabla_metricas(y_test, y_pred_test, y_prob_test, 'Iteración 1 · τ=0.50'),
    tabla_metricas(y2_test, y2_pred_test, y2_prob_test, 'Iteración 2 · τ=0.50'),
    metricas_finales
])
resumen_modelos

In [ ]:
ax = resumen_modelos[['Precision', 'Recall', 'F1', 'AUC']].plot(kind='bar', rot=15)
ax.set_title('Evolución del desempeño del modelo')
ax.set_ylabel('Valor de la métrica')
ax.set_ylim(0, 1)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=4)
plt.tight_layout()
plt.show()

# 9. Conclusiones

1. La **primera iteración** funcionó como baseline, pero su Recall de abandono fue demasiado bajo para una alerta temprana.
2. La **segunda iteración** mejoró la capacidad discriminativa al incorporar información académica, demográfica y contextual.
3. El umbral estándar de **0.50** favorecía la clase mayoritaria y dejaba sin detectar muchos abandonos.
4. Con una partición final **70 % TRAIN / 15 % VALIDATION / 15 % TEST**, VALIDATION seleccionó un umbral de **0.27** mediante F1.
5. En TEST, el modelo final obtuvo aproximadamente **Recall = 0.731**, **F1 = 0.511** y **AUC = 0.673**.
6. De **1.524 abandonos reales**, se detectaron **1.114** y se omitieron **410**. El costo de aumentar la sensibilidad fue generar **1.722 falsas alertas**.
7. La Regresión Logística queda establecida como un **baseline interpretable**. El siguiente paso del proyecto es comparar su desempeño con otros algoritmos de clasificación, manteniendo las mismas reglas de evaluación.

> **Idea central para la exposición:** un Accuracy alto no es suficiente. En este problema importa especialmente cuánto abandono real logra detectar el modelo y qué costo tiene esa detección en falsos positivos.

---
## Anexo · Reproducibilidad

Antes de entregar o subir el notebook al repositorio:

1. Ejecutar **Kernel → Restart Kernel and Run All Cells**.
2. Confirmar que no existan errores.
3. Verificar que las salidas y gráficos queden visibles.
4. Guardar el notebook antes del commit.